In [1]:
import os
import sys
import joblib
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import torch
import plotly.express as px
import pandas as pd
pd.set_option('display.max_colwidth', 100)

from common.eval import Evaluator

evaluator = Evaluator()

/Users/jefferybai/Desktop/Master/BILAB/Master Thesis/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 42


In [2]:
dir = "../experiments/artifacts/finetune"
prefix = "finetune_ngcf_k_all"
user_dps_df = joblib.load(os.path.join(dir, f"user_dps_df.pkl"))
finetune_user_dpms_df = joblib.load(os.path.join(dir, f"{prefix}_user_dpms_df.pkl"))


### EDA on DPS/DPM

In [ ]:
user_dps_df["avg_dps"] = user_dps_df[["actorID_dps", "country_dps", "directorID_dps", "genre_dps"]].mean(axis=1)
user_info_df = finetune_user_dpms_df.merge(user_dps_df, on="userID", how="left")
dimension = "genre"
px.scatter(
    user_info_df,
    x=f"{dimension}_dps", y=f"{dimension}_dpms",
    title="DPS vs. DPMS"
)

In [4]:
exploded_dps_df = pd.concat(
    [
        user_dps_df[["userID", "actorID_dps"]].rename(columns={"actorID_dps": "dps"}).assign(dps_type="actor"),
        user_dps_df[["userID", "country_dps"]].rename(columns={"country_dps": "dps"}).assign(dps_type="country"),
        user_dps_df[["userID", "directorID_dps"]].rename(columns={"directorID_dps": "dps"}).assign(dps_type="director"),
        user_dps_df[["userID", "genre_dps"]].rename(columns={"genre_dps": "dps"}).assign(dps_type="genre"),
    ],
    ignore_index=True
)

In [ ]:
px.histogram(exploded_dps_df, x="dps", facet_col="dps_type", barmode="group", nbins=10, width=800)

In [ ]:
user_dps_df["avg_dps"] = user_dps_df[["actorID_dps", "country_dps", "directorID_dps", "genre_dps"]].mean(axis=1)
px.histogram(user_dps_df, x="avg_dps", nbins=10, width=800, title="Average DPS per User")

In [7]:
user_dps_df.describe()

,userID,actorID_dps,country_dps,directorID_dps,genre_dps,avg_dps
count,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000,2064.000000
mean,1031.500000,0.587783,0.403983,0.587310,0.801343,0.595105
std,595.969798,0.193174,0.159562,0.171116,0.118936,0.123482
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.099763
25%,515.750000,0.459821,0.298241,0.465966,0.756300,0.517554
50%,1031.500000,0.615030,0.391319,0.606004,0.827660,0.619468
75%,1547.250000,0.736483,0.504137,0.716159,0.881848,0.684388
max,2063.000000,1.000000,1.000000,1.000000,1.000000,0.864913


In [8]:
# user_dps_df[user_dps_df["avg_dps"] > 0.8].head() # [56, 192, 217, 246, 260] high
target_users = [56, 192, 217, 246, 260]
display(user_dps_df.loc[
    finetune_user_dpms_df["userID"].isin(target_users), 
    ["userID", "actorID_dps", "country_dps", "directorID_dps", "genre_dps", "avg_dps"]
])

display(finetune_user_dpms_df[finetune_user_dpms_df["userID"].isin(target_users)])

,userID,actorID_dps,country_dps,directorID_dps,genre_dps,avg_dps
56,56,0.846723,0.729528,0.845125,0.890662,0.828010
192,192,0.928065,0.616057,0.934047,0.798865,0.819258
217,217,0.891014,0.677206,0.880842,0.835952,0.821254
246,246,0.774987,0.832309,0.748788,0.931878,0.821990
260,260,0.937631,0.557393,0.906597,0.975163,0.844196


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
56,56,0.108485,0.975494,0.198676,0.809153,0.522952
192,192,0.106304,0.979098,0.074032,0.916650,0.519021
217,217,0.197881,0.978065,0.232503,0.780640,0.547272
246,246,0.060626,0.910799,0.158866,0.876252,0.501636
260,260,0.126060,0.981088,0.140336,0.910564,0.539512


In [9]:
# user_dps_df[user_dps_df["avg_dps"] < 0.20].head() # [148, 542, 561, 803, 1075] low
target_users = [148, 542, 561, 803, 1075]
display(user_dps_df.loc[
    finetune_user_dpms_df["userID"].isin(target_users), 
    ["userID", "actorID_dps", "country_dps", "directorID_dps", "genre_dps", "avg_dps"]
])

display(finetune_user_dpms_df[finetune_user_dpms_df["userID"].isin(target_users)])

,userID,actorID_dps,country_dps,directorID_dps,genre_dps,avg_dps
148,148,0.062947,1.561807e-10,0.180567,0.155537,0.099763
542,542,0.023667,5.199863e-02,0.000346,0.488023,0.141009
561,561,0.068413,3.090926e-01,0.237281,0.111844,0.181658
803,803,0.218277,9.591201e-02,0.290670,0.098017,0.175719
1075,1075,0.040871,3.022096e-10,0.000000,0.363138,0.101002


,userID,actorID_dpms,country_dpms,directorID_dpms,genre_dpms,avg_dpms
148,148,0.033951,0.993884,0.000000,0.312031,0.334967
542,542,0.057140,0.993540,0.196116,0.715226,0.490505
561,561,0.043587,0.966184,0.296319,0.814540,0.530157
803,803,0.059817,0.968926,0.000000,0.932404,0.490287
1075,1075,0.000000,0.993884,0.000000,0.698013,0.422974


### Visualize User Embeddings with t-SNE

In [10]:
from sklearn.manifold import TSNE
tsne = TSNE(n_components=2, perplexity=30, learning_rate=200, max_iter=1000, random_state=42)

In [ ]:
# base model

model_name = "ngcf" # (mt)ngcf_v2_k5, lightgcn_k5
embeddings_dir = "../experiments/embeddings/ngcf/"
original_user_emb = torch.load(f"{embeddings_dir}user_emb.pt") # {model_name}_
original_item_emb = torch.load(f"{embeddings_dir}item_emb.pt") # {model_name}_
print(original_user_emb.size())
print(original_item_emb.size())

original_user_emb_np = original_user_emb.numpy()[:-1] # drop OOV


In [ ]:
original_user_emb_2d = tsne.fit_transform(original_user_emb_np)

px.scatter(
    x=original_user_emb_2d[:, 0], 
    y=original_user_emb_2d[:, 1], 
    color=user_dps_df["avg_dps"], 
    labels={"x": "t-SNE Component 1", "y": "t-SNE Component 2", "color": "Average DPS"},
    title="Original User Embeddings Visualized with t-SNE",
    width=800, height=600
)

In [ ]:
# finetune model
model_name = "ngcf" # (mt)ngcf_v2_k5, lightgcn_k5
embeddings_dir = "../experiments/embeddings/finetune/"
finetune_user_emb = torch.load(f"{embeddings_dir}{model_name}_user_emb.pt") # {model_name}_
finetune_item_emb = torch.load(f"{embeddings_dir}{model_name}_item_emb.pt") # {model_name}_
print(finetune_user_emb.size())
print(finetune_item_emb.size())

finetune_user_emb_np = finetune_user_emb.numpy()[:-1] # drop OOV

torch.Size([2065, 256])
torch.Size([8397, 256])


In [ ]:
finetune_user_emb_2d = tsne.fit_transform(finetune_user_emb_np)

px.scatter(
    x=finetune_user_emb_2d[:, 0],
    y=finetune_user_emb_2d[:, 1],
    color=user_dps_df["avg_dps"],
    labels={"x": "t-SNE Component 1", "y": "t-SNE Component 2", "color": "Average DPS"},
    title="Finetuned User Embeddings Visualized with t-SNE",
    width=800, height=600
)

### Qualitative Evaluate Recommendation Results

In [ ]:
file_path = "../datasets/hetrec2011-movielens-2k-v2/movies.dat"
movie_name_df = pd.read_table(file_path, encoding="latin")
movie_name_df.head(1)

,id,title,imdbID,spanishTitle,imdbPictureURL,year,rtID,rtAllCriticsRating,rtAllCriticsNumReviews,rtAllCriticsNumFresh,...,rtAllCriticsScore,rtTopCriticsRating,rtTopCriticsNumReviews,rtTopCriticsNumFresh,rtTopCriticsNumRotten,rtTopCriticsScore,rtAudienceRating,rtAudienceNumRatings,rtAudienceScore,rtPictureURL
0,1,Toy story,114709,Toy story (juguetes),http://ia.media-imdb.com/images/M/MV5BMTMwNDU0NTY2Nl5BMl5BanBnXkFtZTcwOTUxOTM5Mw@@._V1._SX214_CR...,1995,toy_story,9,73,73,...,100,8.5,17,17,0,100,3.7,102338,81,http://content7.flixster.com/movie/10/93/63/10936393_det.jpg


#### find users with high/low dps on genre

In [ ]:
original_dir = "../experiments/artifacts/ngcf"
original_prefix = "ngcf_k_all"
original_user_dpms_df = joblib.load(os.path.join(original_dir, f"{original_prefix}_user_dpms_df.pkl"))

dir = "../experiments/artifacts/finetune"
prefix = "finetune_ngcf_k_all"
feature_engineer = joblib.load(os.path.join(dir, f"feature_engineer.pkl"))
finetune_user_dpms_df = joblib.load(os.path.join(dir, f"{prefix}_user_dpms_df.pkl"))
user_info_df = finetune_user_dpms_df.merge(user_dps_df, on="userID", how="inner").merge(original_user_dpms_df, on="userID", how="inner", suffixes=("", "_original"))
user_info_df["genre_improvement"] = user_info_df["genre_dpms"] - user_info_df["genre_dpms_original"]
user_info_df[["userID", "genre_dpms", "genre_dpms_original", "genre_improvement"]]

print("user with high dp score on genre:", user_info_df[(user_info_df["genre_dps"] > 0.97)].sort_values("genre_improvement", ascending=False).head(5)["userID"].tolist())
print("user with low dp score on genre:", user_info_df[(user_info_df["genre_dps"] < 0.4)].sort_values("genre_improvement", ascending=False).head(5)["userID"].tolist())

user with high dp score on genre: [952, 1203, 260, 1567, 302]
user with low dp score on genre: [2057, 803, 319, 930, 1075]


In [ ]:
# high dps target user (260)
HIGH_USER_ID = 952
high_user_genre_pd = user_info_df.loc[user_info_df["userID"] == HIGH_USER_ID, "genre_wvec"]
high_user_genre_pd = np.array(high_user_genre_pd.tolist()).squeeze(0)[:-1] # skip oov token
high_user_genre_pd = np.round(high_user_genre_pd / high_user_genre_pd.sum(), 3)
genre_tags = [
    feature_engineer.idx2vocab["genre"][i]
    for i in range(2, len(feature_engineer.idx2vocab["genre"]) - 1)
] # skip pad and rare tokens
high_user_genre_df = pd.DataFrame(
    {"genre": genre_tags, "weight": high_user_genre_pd}, 
    columns=["genre", "weight"]
).sort_values(by="weight", ascending=False)

high_user_genre_df

,genre,weight
7,Drama,0.155
4,Comedy,0.104
16,Thriller,0.097
0,Action,0.070
10,Horror,0.069
14,Romance,0.068
5,Crime,0.063
1,Adventure,0.061
13,Mystery,0.054
2,Animation,0.052


In [ ]:
px.bar(
    high_user_genre_df, 
    x="genre", 
    y="weight", 
    title=f"Genre Preference Distribution for High DPS User ({HIGH_USER_ID})",
    labels={"genre": "Genre", "weight": "Preference Weight"},
    width=800, height=400
)

In [ ]:
# low dps target user
LOW_USER_ID = 803
low_user_genre_pd = user_info_df.loc[user_info_df["userID"] == LOW_USER_ID, "genre_wvec"]
low_user_genre_pd = np.array(low_user_genre_pd.tolist()).squeeze(0)[:-1] # skip oov token
low_user_genre_pd = np.round(low_user_genre_pd / low_user_genre_pd.sum(), 3)
genre_tags = [
    feature_engineer.idx2vocab["genre"][i]
    for i in range(2, len(feature_engineer.idx2vocab["genre"]) - 1)
] # skip pad and rare tokens

low_user_genre_df = pd.DataFrame(
    {"genre": genre_tags, "weight": low_user_genre_pd}, 
    columns=["genre", "weight"]
).sort_values(by="weight", ascending=False)

low_user_genre_df

,genre,weight
7,Drama,0.329
4,Comedy,0.251
14,Romance,0.228
16,Thriller,0.096
3,Children,0.048
13,Mystery,0.048
0,Action,0.000
11,IMAX,0.000
17,War,0.000
15,Sci-Fi,0.000


In [ ]:
px.bar(
    low_user_genre_df, 
    x="genre", 
    y="weight", 
    title=f"Genre Preference Distribution for Low DPS User ({LOW_USER_ID})",
    labels={"genre": "Genre", "weight": "Preference Weight"},
    width=800, height=400
)

#### Original Recommendation Results

In [ ]:
dir = "../experiments/artifacts/ngcf"
prefix = "ngcf_k_all"
original_eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))

In [ ]:
processed_original_eval_df = evaluator.process_eval_df(
    eval_df=original_eval_df, 
    feature_engineer=feature_engineer,
    k=10,
    rare_threshold=0,
    encoded=False
)

processed_original_eval_df["genre"] = processed_original_eval_df["genre"].apply(lambda x: sorted(set(x) - {"[PAD]"}))
processed_original_eval_df["userID"] = processed_original_eval_df["userID"].map(feature_engineer.vocab2idx["userID"]).astype(int)
processed_original_eval_df = processed_original_eval_df[["userID", "movieID", "rank", "genre"]]

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!


In [ ]:
# high dps user / low dps user
display(
    (
        processed_original_eval_df[processed_original_eval_df["userID"] == HIGH_USER_ID]
        .merge(movie_name_df[["id", "title"]], left_on="movieID", right_on="id", how="left")
        [["userID", "title", "genre", "rank"]]
    )
)
tmp_high_df = processed_original_eval_df.loc[processed_original_eval_df["userID"] == HIGH_USER_ID, "genre"]
high_genre_set = set()
for genres in tmp_high_df.tolist():
    high_genre_set.update(set(genres))
print(sorted(high_genre_set))
print(len(high_genre_set))

display(
    (
        processed_original_eval_df[processed_original_eval_df["userID"] == LOW_USER_ID]
        .merge(movie_name_df[["id", "title"]], left_on="movieID", right_on="id", how="left")
        [["userID", "title", "genre", "rank"]]
    )
)
tmp_low_df = processed_original_eval_df.loc[processed_original_eval_df["userID"] == LOW_USER_ID, "genre"]
low_genre_set = set()
for genres in tmp_low_df.tolist():
    low_genre_set.update(set(genres))
print(sorted(low_genre_set))
print(len(low_genre_set))


,userID,title,genre,rank
0,952,In the Name of the Father,[Drama],1
1,952,Good Will Hunting,"[Drama, Romance]",2
2,952,2001: A Space Odyssey,"[Adventure, Sci-Fi]",3
3,952,The Shawshank Redemption,[Drama],4
4,952,The Breakfast Club,"[Comedy, Drama]",5
5,952,The Sting,"[Comedy, Crime]",6
6,952,Mar adentro,[Drama],7
7,952,Pulp Fiction,"[Comedy, Crime, Drama]",8
8,952,About a Boy,"[Comedy, Drama]",9
9,952,American Beauty,[Drama],10


['Adventure', 'Comedy', 'Crime', 'Drama', 'Romance', 'Sci-Fi']
6


,userID,title,genre,rank
0,803,Far from Heaven,"[Drama, Romance]",1
1,803,Yûgiô Duel Monsters: Hikari no pyramid,"[Action, Adventure, Animation, Fantasy]",2
2,803,Rustlers' Rhapsody,"[Comedy, Western]",3
3,803,Bittersweet Motel,[Documentary],4
4,803,Session 9,"[Horror, Thriller]",5
5,803,Les rivières pourpres,"[Crime, Drama, Mystery, Thriller]",6
6,803,Ging chat goo si,"[Action, Comedy, Crime, Thriller]",7
7,803,Sky High,"[Action, Adventure, Children, Comedy]",8
8,803,Harold & Kumar Go to White Castle,[Comedy],9
9,803,Les poupées russes,"[Comedy, Romance]",10


['Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Horror', 'Mystery', 'Romance', 'Thriller', 'Western']
14


#### Finetuned Recommnedation Results

In [ ]:
dir = "../experiments/artifacts/finetune"
prefix = "finetune_ngcf_k_all"
finetune_eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))

In [ ]:
# process finetuned rec results
processed_finetune_eval_df = evaluator.process_eval_df(
    eval_df=finetune_eval_df, 
    feature_engineer=feature_engineer,
    k=10,
    rare_threshold=0,
    encoded=False
)

processed_finetune_eval_df["genre"] = processed_finetune_eval_df["genre"].apply(lambda x: sorted(set(x) - {"[PAD]"}))
processed_finetune_eval_df["userID"] = processed_finetune_eval_df["userID"].map(feature_engineer.vocab2idx["userID"]).astype(int)
processed_finetune_eval_df = processed_finetune_eval_df[["userID", "movieID", "rank", "genre"]]

candidate item pool size: 10
exploded 2064
extracting item features...
merging features...
interaction data count before merging: 20640
interaction data count after merging: 20640
done!


In [ ]:
# high dps user 260 / low dps user 803
display(
    (
        processed_finetune_eval_df[processed_finetune_eval_df["userID"] == HIGH_USER_ID]
        .merge(movie_name_df[["id", "title"]], left_on="movieID", right_on="id", how="left")
        [["userID", "title", "genre", "rank"]]
    )
)
tmp_high_df = processed_finetune_eval_df.loc[processed_finetune_eval_df["userID"] == HIGH_USER_ID, "genre"]
high_genre_set = set()
for genres in tmp_high_df.tolist():
    high_genre_set.update(set(genres))
print(sorted(high_genre_set))
print(len(high_genre_set))

display(
    (
        processed_finetune_eval_df[processed_finetune_eval_df["userID"] == LOW_USER_ID]
        .merge(movie_name_df[["id", "title"]], left_on="movieID", right_on="id", how="left")
        [["userID", "title", "genre", "rank"]]
    )
)
tmp_low_df = processed_finetune_eval_df.loc[processed_finetune_eval_df["userID"] == LOW_USER_ID, "genre"]
low_genre_set = set()
for genres in tmp_low_df.tolist():
    low_genre_set.update(set(genres))
print(sorted(low_genre_set))
print(len(low_genre_set))


,userID,title,genre,rank
0,952,Pulp Fiction,"[Comedy, Crime, Drama]",1
1,952,The Shawshank Redemption,[Drama],2
2,952,American Beauty,[Drama],3
3,952,Schindler's List,"[Drama, War]",4
4,952,Crash,"[Crime, Drama]",5
5,952,Walk the Line,"[Drama, Musical, Romance]",6
6,952,Jaws,"[Action, Horror]",7
7,952,Good Will Hunting,"[Drama, Romance]",8
8,952,Blade Runner,"[Adventure, Drama, Film-Noir, Sci-Fi, Thriller]",9
9,952,Kill Bill: Vol. 2,"[Action, Drama, Thriller]",10


['Action', 'Adventure', 'Comedy', 'Crime', 'Drama', 'Film-Noir', 'Horror', 'Musical', 'Romance', 'Sci-Fi', 'Thriller', 'War']
12


,userID,title,genre,rank
0,803,The Three Burials of Melquiades Estrada,"[Action, Adventure, Crime, Drama, Western]",1
1,803,Harold & Kumar Go to White Castle,[Comedy],2
2,803,Far from Heaven,"[Drama, Romance]",3
3,803,Last Holiday,[Comedy],4
4,803,Wordplay,[Documentary],5
5,803,L'âge d'or,"[Comedy, Drama, Fantasy, Romance]",6
6,803,The Break-Up,"[Comedy, Drama, Romance]",7
7,803,Laurel Canyon,[Drama],8
8,803,Flawless,[Drama],9
9,803,Malèna,"[Comedy, Drama, Romance, War]",10


['Action', 'Adventure', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Romance', 'War', 'Western']
10
